# Connect the forgetting mechanism

Use the original saved campaign on your GH200. Run Configure, then Run / resume. The detached experiment stays quiet; refresh status or results when you want. Defaults allow 11.5 hours per session. See README.md for design, resource requirements and interpretation.

Keep all three Python modules beside this notebook. Existing experiments are not overwritten.

## Configure

In [ ]:
from pathlib import Path
import importlib, json
import mechanism_bridge as study
importlib.reload(study)
SESSION = Path.cwd() / "bridge_session.json"
saved = json.loads(SESSION.read_text()) if SESSION.exists() else {}
SOURCE = Path(saved["source"]) if saved.get("source") else study.discover_source()
# If automatic detection fails, set the ORIGINAL campaign directory here:
# SOURCE = Path("/home/ubuntu/1/runs/olmo_association_v1")
if SOURCE is None:
    raise FileNotFoundError("Set SOURCE above to the original directory with config.json and seed*/fork*/fork.pt; a results ZIP is insufficient.")
OUTPUT = Path(saved["output"]) if saved.get("source") == str(SOURCE) else SOURCE.parent / "olmo_mechanism_bridge_v1"
SETTINGS = study.defaults(SOURCE, OUTPUT)
SETTINGS.update(hours=11.5, device="cuda:0")
# Scientific options (changing these creates a new output directory):
# SETTINGS["patch_layers"] = []   # All layers; defaults prioritize indices 0, 1, 9.
# SETTINGS["events"] = [21, 20]   # Defaults also include 47/46 if available.
# SETTINGS["fractions"] = [.625, .575, .55, .6, .5, .675, 1.]
study.validate(SETTINGS)
print("Original source:", SOURCE)
print("Results:", SETTINGS["output"])
print("Jobs:", len(study.plan(SETTINGS)), "— multiple sessions may be needed")
print("Reserve at least 150–200 GB of additional disk for a full 1B campaign.")
def remember():
    study.atomic_json({"source": str(SOURCE), "output": SETTINGS["output"]}, SESSION)


## Run / resume
Rerunning this cell does not launch a duplicate worker. After a time-budget pause, it grants another session.

In [ ]:
OUTPUT = study.launch(SETTINGS)
remember()
print("Run directory:", OUTPUT)
print("Rerun Refresh status to see progress.")

## Refresh status
Run this cell manually whenever you want an update.

In [ ]:
_ = study.refresh(SETTINGS["output"])

## Show results
Partial jobs may have many saved records before a complete four-way comparison is reportable.

In [ ]:
_ = study.report(SETTINGS["output"], display=True)
from IPython.display import display, Image, FileLink
for name in ["repair_interactions.png", "continuations.png", "interactions.png"]:
    file = Path(SETTINGS["output"]) / name
    if file.exists(): display(Image(filename=str(file)))
# Refresh the share ZIP and place a downloadable copy beside this notebook.
import shutil
archive = study.export(SETTINGS["output"], SETTINGS["export_arrays"])
download = Path.cwd() / "bridge_results_share.zip"
shutil.copy2(archive, download)
print("Share ZIP:", download)
display(FileLink(download.name))


## Optional controls
Choose stop, resume, restart, export or status. Restart creates a fresh folder and preserves previous results. Default status makes Run All safe.

In [ ]:
ACTION = "status"
if ACTION == "stop":
    print(study.stop(SETTINGS["output"]))
elif ACTION == "resume":
    OUTPUT = study.launch(SETTINGS); remember()
elif ACTION == "restart":
    OUTPUT = study.restart(SETTINGS); remember()
elif ACTION == "export":
    study.report(SETTINGS["output"])
    import shutil
    archive = study.export(SETTINGS["output"], SETTINGS["export_arrays"])
    download = Path.cwd() / "bridge_results_share.zip"
    shutil.copy2(archive, download)
    print("Share ZIP:", download)
elif ACTION != "status":
    raise ValueError("Choose status, stop, resume, restart or export.")
_ = study.refresh(SETTINGS["output"])